In [136]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [137]:
import sys
sys.path.append("../")

In [138]:
import pandas as pd
import matplotlib.pyplot as plt
import glob

import random

from phonetics import utils as u
from phonetics import plots as plots
from pydub import AudioSegment, effects 

In [139]:
root = '/Users/tomasandrade/Documents/BSC/ICHOIR/datasets/maria_voice/LABELED_FULL/'

In [140]:
lab_file = f'{root}/ES_milagro.lab'
wav_file = f'{root}/ES_milagro.wav'

In [141]:
df_algn = u.df_alignments_from_lab_file(lab_file, add_transitions=False)

In [142]:
df_algn["next_phone"] = df_algn["phone_base"].shift(-1)
df_algn["syllable"] = df_algn["phone_base"] + df_algn["next_phone"]
df_algn = df_algn.iloc[:-1, :] 

In [143]:
def extract_segment(audio, syl = 'ka'):

    df = make_syllable_df(syl)

    start = int(df['start'].iloc[0]*1000)
    end   = int(df['end'].iloc[1]*1000)

    segment = audio[start:end]

    return segment

def make_syllable_df(syl):
    idxs = get_idxs(syl = syl)
    idx = random.choice(idxs)
    df = df_algn[idx:idx+2].reset_index(drop=True)
    return df

def get_idxs(syl = 'ka'):
    idxs = df_algn[df_algn['syllable'] == syl].index
    return idxs

In [152]:
song = AudioSegment.from_file(wav_file, format='wav')

In [153]:
def make_word(song, syls = ['ka', 'ta', 'la']):
    segments = []
    for syl in syls:
        seg = extract_segment(song, syl = syl)
        segments.append(seg)

    output_audio = sum(segments)
    output_audio= effects.normalize(output_audio)  

    return output_audio

In [154]:
segments = []
for i in range(10):
    word = make_word(song, syls = ['ka', 'ta', 'la'])
    segments.append(word)
    silence = AudioSegment.silent(duration=500)
    segments.append(silence)

output_audio = sum(segments)

In [155]:
output_audio